# Mutation Analysis and Visualization

This notebook demonstrates mutation pattern analysis and visualization techniques.

## Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pipelines import VariantCaller, MutationAnnotator
from models import DriverMutationClassifier

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Load Mutation Data

In [ ]:
# Example mutation data
mutations = pd.DataFrame({
    'gene': ['TP53', 'KRAS', 'EGFR', 'BRAF', 'PIK3CA', 'PTEN', 'TP53', 'APC'],
    'chromosome': ['chr17', 'chr12', 'chr7', 'chr7', 'chr3', 'chr10', 'chr17', 'chr5'],
    'position': [7577548, 25398284, 55249071, 140453136, 178936091, 89692905, 7578406, 112175770],
    'ref': ['C', 'G', 'C', 'T', 'G', 'A', 'G', 'C'],
    'alt': ['T', 'A', 'T', 'A', 'A', 'G', 'A', 'T'],
    'variant_allele_frequency': [0.45, 0.38, 0.52, 0.41, 0.35, 0.48, 0.51, 0.33]
})

print(f"Loaded {len(mutations)} mutations")
display(mutations.head())

## Annotate Mutations

In [ ]:
annotator = MutationAnnotator()
annotated_mutations = annotator.annotate_mutations(mutations)

print(f"Cancer genes found: {annotated_mutations['is_cancer_gene'].sum()}")
display(annotated_mutations)

## Visualize Mutation Distribution

In [ ]:
# Mutations per gene
gene_counts = annotated_mutations['gene'].value_counts()

plt.figure(figsize=(10, 6))
gene_counts.head(10).plot(kind='barh', color='steelblue')
plt.title('Top 10 Mutated Genes')
plt.xlabel('Number of Mutations')
plt.ylabel('Gene')
plt.tight_layout()
plt.show()

In [ ]:
# Variant Allele Frequency distribution
plt.figure(figsize=(10, 6))
plt.hist(annotated_mutations['variant_allele_frequency'], bins=20, color='coral', edgecolor='black')
plt.title('Variant Allele Frequency Distribution')
plt.xlabel('Variant Allele Frequency')
plt.ylabel('Count')
plt.axvline(0.5, color='red', linestyle='--', label='Expected (heterozygous)')
plt.legend()
plt.tight_layout()
plt.show()

## Identify Driver Mutations

In [ ]:
driver_mutations = annotator.prioritize_driver_mutations(annotated_mutations)

print(f"Identified {len(driver_mutations)} potential driver mutations:")
display(driver_mutations[['gene', 'chromosome', 'position', 'ref', 'alt', 'driver_score']])

## Mutation Spectrum Analysis

In [ ]:
# Create mutation types
annotated_mutations['mutation_type'] = annotated_mutations.apply(
    lambda row: f"{row['ref']}>{row['alt']}", axis=1
)

mutation_spectrum = annotated_mutations['mutation_type'].value_counts()

plt.figure(figsize=(10, 6))
mutation_spectrum.plot(kind='bar', color='mediumseagreen')
plt.title('Mutation Spectrum')
plt.xlabel('Mutation Type')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
print("Mutation Analysis Summary:")
print(f"  Total mutations: {len(annotated_mutations)}")
print(f"  Unique genes: {annotated_mutations['gene'].nunique()}")
print(f"  Cancer genes: {annotated_mutations['is_cancer_gene'].sum()}")
print(f"  Driver candidates: {len(driver_mutations)}")
print(f"\nMost frequently mutated genes:")
print(gene_counts.head())